# Phase 3 — Data Preparation

## 3.1 Preparation Strategy and Modeling Assumptions

**Objective:** Design a reproducible, leakage-aware data preparation workflow for transaction-level fraud prediction using the BankSim synthetic dataset.

The intended prediction task is to estimate the probability that a submitted transaction is fraudulent, using information assumed to be available at the time of assessment.

The preparation strategy follows the findings of the completed CRISP-DM Data Understanding phase.

The principal design requirements are:

1. Preserve the original raw dataset without modification.
2. Define candidate predictors and their assumed prediction-time availability.
3. Establish chronological training, validation, and test partitions before fitting data-dependent transformations.
4. Apply consistent preprocessing across partitions while preventing information leakage.
5. Develop a simple baseline feature set before introducing customer- or merchant-level historical features.
6. Construct historical features using only information available before each prediction.
7. Verify the integrity, consistency, and reproducibility of the prepared datasets.

**Provisional evaluation design:** Use complete simulation steps for chronological partitioning, initially considering steps 0–125 for training, 126–152 for validation, and 153–179 for testing. The partition will be verified before implementation.

**Initial feature strategy:** Begin with a limited set of transaction-time attributes. Exclude the target from predictors, omit constant postal-code variables, and defer raw customer/merchant identifiers and historical aggregates until their modeling role is explicitly justified.

No transformations are performed at this stage. This section establishes the methodological foundation for subsequent preparation activities.

## 3.2 Candidate Feature Selection and Prediction-Time Availability

**Objective:** Define an initial set of predictors for transaction-level fraud risk assessment, document the rationale for including or excluding each original variable, and establish the information-availability assumptions required for leakage-safe modeling.

The intended prediction scenario is the assessment of a submitted transaction before its true fraud outcome is known. Only attributes assumed to be available at that time may be used as predictors.

Because BankSim is synthetic and does not specify a complete operational transaction-processing workflow, prediction-time availability cannot be independently verified. The following decisions represent explicit modeling assumptions.

### Initial Baseline Feature Set

The first baseline model will use:

- `amount`: transaction amount.
- `category`: transaction category.

These variables are selected because they describe the submitted transaction, exhibit meaningful variation, and are assumed to be available at the time of risk assessment.

The baseline intentionally excludes customer and merchant identifiers, demographic variables, and the simulation-step index as direct predictors. This allows an initial assessment of fraud detection using transaction-level information without relying on identity-specific memorization or synthetic temporal patterns.

### Treatment of Remaining Variables

| Variable | Decision | Justification |
|---|---|---|
| `step` | Retain for partitioning; defer as predictor | Provides simulation ordering but not actual timestamps. |
| `customer` | Retain for future feature engineering | May support historical customer behavior features, subject to temporal availability constraints. |
| `merchant` | Retain for future feature engineering | May support merchant-specific historical features and generalization experiments. |
| `age` | Defer | Requires a separate assessment of interpretation, relevance, and fairness. |
| `gender` | Defer | Contains demographic and customer-type codes; requires separate consideration of governance and operational relevance. |
| `zipcodeOri` | Exclude | Constant across all observations. |
| `zipMerchant` | Exclude | Constant across all observations. |
| `fraud` | Target only | Represents the outcome to be predicted and must never enter the predictor matrix. |

The deferred variables will remain available in the working dataset for subsequent controlled experiments. Their exclusion from the initial baseline does not imply permanent removal from the project.

### Prediction-Time and Leakage Constraints

The initial baseline assumes that transaction amount and category are available when the transaction is submitted.

The following restrictions apply:

1. The `fraud` label must be separated from the predictors.
2. The `step` variable must be retained for chronological partitioning but excluded from the initial predictor set.
3. No fraud statistics calculated using future or held-out observations may be used as predictor values.
4. Data-dependent preprocessing, including categorical encoding where applicable, must be fitted on the appropriate training partition.
5. Validation and test observations must not influence preprocessing parameters or model selection outside their designated roles.
6. Customer- and merchant-level historical features will be considered only after defining an explicit past-information policy.

### Decision

The initial baseline predictor set is:

`X_baseline = ["amount", "category"]`

The target variable is:

`y = "fraud"`

The simulation-step variable and entity identifiers will be retained as supporting information for partitioning and possible later feature engineering.

No transformations, encodings, or data filtering are performed in this subsection.

In [6]:
# ============================================================
# 3.2 — Candidate Feature Selection
# ============================================================

# Initial baseline predictors
BASELINE_FEATURES = ["amount", "category"]

# Prediction target
TARGET = "fraud"

# Chronological partitioning variable
TIME_COLUMN = "step"

# Entity identifiers retained for possible future feature engineering
ENTITY_COLUMNS = ["customer", "merchant"]

# Variables excluded from the initial baseline
DEFERRED_FEATURES = [
    "step", "customer", "merchant", "age", "gender"
]

CONSTANT_COLUMNS = ["zipcodeOri", "zipMerchant"]

# Verify configuration consistency
assert TARGET not in BASELINE_FEATURES
assert TIME_COLUMN not in BASELINE_FEATURES

assert len(BASELINE_FEATURES) == len(set(BASELINE_FEATURES))

assert not (
    set(BASELINE_FEATURES) & set(DEFERRED_FEATURES)
)

assert not (
    set(BASELINE_FEATURES) & set(CONSTANT_COLUMNS)
)

print("Baseline predictors:", BASELINE_FEATURES)
print("Target:", TARGET)
print("Time column:", TIME_COLUMN)
print("Entity identifiers:", ENTITY_COLUMNS)
print("Deferred features:", DEFERRED_FEATURES)
print("Constant columns:", CONSTANT_COLUMNS)

print("\nFeature configuration checks passed.")

Baseline predictors: ['amount', 'category']
Target: fraud
Time column: step
Entity identifiers: ['customer', 'merchant']
Deferred features: ['step', 'customer', 'merchant', 'age', 'gender']
Constant columns: ['zipcodeOri', 'zipMerchant']

Feature configuration checks passed.


## 3.3 Chronological Data Partitioning

**Objective:** Establish a reproducible chronological training, validation, and test partition for the BankSim dataset while preserving the temporal ordering of simulation steps and preventing information leakage across partitions.

The `step` variable ranges from 0 to 179 and provides the available simulation ordering. Because individual transaction timestamps are not provided, complete simulation steps will be assigned to each partition without splitting observations from the same step across partitions.

The initial proposed partition is:

| Partition | Simulation Steps | Number of Steps | Proportion of Steps |
|---|---|---:|---:|
| Training | 0–125 | 126 | 70% |
| Validation | 126–152 | 27 | 15% |
| Test | 153–179 | 27 | 15% |

These percentages refer to simulation steps, not necessarily to the number of transactions.

Before finalizing the partition, we will verify its coverage, disjointness, chronological ordering, transaction counts, and fraud prevalence.

The test partition will be reserved for final evaluation and must not be used to select models, tune hyperparameters, or determine preprocessing decisions.

No preprocessing transformations or feature engineering will be performed during this initial partition assessment.

In [9]:
# ============================================================
# 3.3 — Chronological Data Partitioning
# ============================================================

import pandas as pd
from pathlib import Path

# Load the original dataset without modifying the source file
DATA_PATH = Path("../data/raw/bs140513_032310.csv")

df = pd.read_csv(DATA_PATH)

# Verify required columns and observed temporal coverage
required_columns = (
    BASELINE_FEATURES
    + [TARGET, TIME_COLUMN]
    + ENTITY_COLUMNS
)

assert set(required_columns).issubset(df.columns)
assert df[TIME_COLUMN].min() == 0
assert df[TIME_COLUMN].max() == 179
assert df[TIME_COLUMN].nunique() == 180

print("Dataset shape:", df.shape)

# Define provisional chronological boundaries
TRAIN_END_STEP = 125
VALID_END_STEP = 152

train_df = df.loc[df[TIME_COLUMN] <= TRAIN_END_STEP].copy()

valid_df = df.loc[
    (df[TIME_COLUMN] > TRAIN_END_STEP)
    & (df[TIME_COLUMN] <= VALID_END_STEP)
].copy()

test_df = df.loc[
    df[TIME_COLUMN] > VALID_END_STEP
].copy()

# Verify partition integrity
assert len(train_df) + len(valid_df) + len(test_df) == len(df)

assert train_df[TIME_COLUMN].max() < valid_df[TIME_COLUMN].min()
assert valid_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()

assert set(train_df.index).isdisjoint(valid_df.index)
assert set(train_df.index).isdisjoint(test_df.index)
assert set(valid_df.index).isdisjoint(test_df.index)

# Summarize the partition distributions
partition_summary = pd.DataFrame({
    "partition": ["Training", "Validation", "Test"],
    "transactions": [
        len(train_df), len(valid_df), len(test_df)
    ],
    "fraud_count": [
        train_df[TARGET].sum(),
        valid_df[TARGET].sum(),
        test_df[TARGET].sum()
    ],
    "step_count": [
        train_df[TIME_COLUMN].nunique(),
        valid_df[TIME_COLUMN].nunique(),
        test_df[TIME_COLUMN].nunique()
    ]
})

partition_summary["transaction_pct"] = (
    100 * partition_summary["transactions"] / len(df)
)

partition_summary["fraud_rate_pct"] = (
    100 * partition_summary["fraud_count"]
    / partition_summary["transactions"]
)

print("\nChronological partition summary:")
print(partition_summary.to_string(index=False))

print("\nPartition integrity checks passed.")

Dataset shape: (594643, 10)

Chronological partition summary:
 partition  transactions  fraud_count  step_count  transaction_pct  fraud_rate_pct
  Training        396332         5040         126        66.650410        1.271661
Validation         98194         1080          27        16.513101        1.099864
      Test        100117         1080          27        16.836488        1.078738

Partition integrity checks passed.


**Findings and Partition Decision**

The proposed chronological partition was successfully implemented and verified using the original BankSim dataset.

The training partition contains 396,332 transactions across simulation steps 0–125, including 5,040 fraudulent transactions (1.2717%).

The validation partition contains 98,194 transactions across steps 126–152, including 1,080 fraudulent transactions (1.0999%).

The test partition contains 100,117 transactions across steps 153–179, including 1,080 fraudulent transactions (1.0787%).

The corresponding proportions of all transactions are approximately 66.65%, 16.51%, and 16.84%, respectively. These differ from the intended 70%/15%/15% allocation of simulation steps because transaction volume is not constant across steps.

All partition integrity checks passed. The partitions collectively cover the complete dataset, have disjoint row indices, and preserve chronological ordering.

The decreasing fraud prevalence across partitions is consistent with the synthetic dataset's fixed count of 40 fraudulent transactions per step and varying transaction volumes.

**Decision:** Retain the proposed chronological boundaries for the initial experimental design. No stratification, random reshuffling, resampling, or balancing is applied to the validation or test partitions.

The test partition is reserved for final evaluation. Model development, preprocessing decisions, and hyperparameter selection will use the training and validation partitions according to their designated roles.

This partition establishes the foundation for leakage-aware data preparation, although the prevention of feature-engineering and preprocessing leakage must be verified separately.

## 3.4 Baseline Data Preparation and Preprocessing

**Objective:** Implement and verify a reproducible preprocessing pipeline for the initial transaction-level fraud prediction baseline, using the chronological partitions established in Section 3.3.

The baseline predictor set consists of:

- `amount`: numerical transaction amount.
- `category`: categorical transaction type.

The preprocessing strategy is:

1. Preserve `amount` as a numerical predictor and standardize it using `StandardScaler`.
2. Encode `category` using `OneHotEncoder`, with explicit handling of categories not encountered during training.
3. Combine these transformations using `ColumnTransformer`.
4. Fit preprocessing transformations exclusively on the training partition.
5. Apply the fitted transformations to validation and test partitions without refitting.

Standardization is adopted as an initial configuration suitable for scale-sensitive models, including logistic regression. It does not eliminate the right-skewness or extreme values previously identified in transaction amounts.

The use of `handle_unknown="ignore"` allows the categorical encoder to process previously unseen categories without failing. Unseen categories will receive zero values across the corresponding one-hot columns.

No resampling, feature selection based on target outcomes, or model training is performed in this subsection.

The original dataset remains unchanged.

In [14]:
# ============================================================
# 3.4 — Baseline Data Preparation and Preprocessing
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Define numerical and categorical predictors
NUMERIC_FEATURES = ["amount"]
CATEGORICAL_FEATURES = ["category"]

# Verify the preprocessing configuration
assert set(NUMERIC_FEATURES + CATEGORICAL_FEATURES) == set(BASELINE_FEATURES)

# Separate predictors and target for each chronological partition
X_train = train_df[BASELINE_FEATURES].copy()
X_valid = valid_df[BASELINE_FEATURES].copy()
X_test = test_df[BASELINE_FEATURES].copy()

y_train = train_df[TARGET].copy()
y_valid = valid_df[TARGET].copy()
y_test = test_df[TARGET].copy()

# Define preprocessing transformations
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, NUMERIC_FEATURES),
        ("categorical", categorical_transformer, CATEGORICAL_FEATURES)
    ],
    remainder="drop"
)

# Fit exclusively on the training partition
X_train_prepared = preprocessor.fit_transform(X_train)

# Transform validation and test without refitting
X_valid_prepared = preprocessor.transform(X_valid)
X_test_prepared = preprocessor.transform(X_test)

# Verify output dimensions and row preservation
assert X_train_prepared.shape[0] == len(train_df)
assert X_valid_prepared.shape[0] == len(valid_df)
assert X_test_prepared.shape[0] == len(test_df)

assert (
    X_train_prepared.shape[1]
    == X_valid_prepared.shape[1]
    == X_test_prepared.shape[1]
)

print("Training matrix shape:", X_train_prepared.shape)
print("Validation matrix shape:", X_valid_prepared.shape)
print("Test matrix shape:", X_test_prepared.shape)

print("\nNumber of prepared features:",
      X_train_prepared.shape[1])

print("\nPrepared feature names:")
print(preprocessor.get_feature_names_out())

print("\nPreprocessing checks passed.")

Training matrix shape: (396332, 16)
Validation matrix shape: (98194, 16)
Test matrix shape: (100117, 16)

Number of prepared features: 16

Prepared feature names:
['numeric__amount' "categorical__category_'es_barsandrestaurants'"
 "categorical__category_'es_contents'"
 "categorical__category_'es_fashion'" "categorical__category_'es_food'"
 "categorical__category_'es_health'" "categorical__category_'es_home'"
 "categorical__category_'es_hotelservices'"
 "categorical__category_'es_hyper'" "categorical__category_'es_leisure'"
 "categorical__category_'es_otherservices'"
 "categorical__category_'es_sportsandtoys'"
 "categorical__category_'es_tech'"
 "categorical__category_'es_transportation'"
 "categorical__category_'es_travel'"
 "categorical__category_'es_wellnessandbeauty'"]

Preprocessing checks passed.


**Findings and Interpretation**

The baseline preprocessing pipeline was successfully fitted using the chronological training partition and applied to the validation and test partitions without refitting.

The resulting feature matrices contain 396,332 training observations, 98,194 validation observations, and 100,117 test observations. All original observations were preserved.

The preprocessing produces 16 features: one standardized numerical transaction-amount feature and 15 one-hot-encoded transaction-category indicators.

The number of encoded categorical features is consistent with the 15 transaction categories identified during Data Understanding.

All three prepared matrices have identical feature dimensions, and the generated feature names are consistent with the specified preprocessing configuration.

The preprocessing workflow separates predictor and target variables and fits transformation parameters exclusively on training data. This prevents validation and test observations from influencing the fitted scaler and categorical encoder.

Additional checks of numerical finiteness, scaling behavior, and encoded category coverage will be performed before considering the prepared matrices ready for modeling.

No observations were removed, and the original raw dataset remains unchanged.

## 3.5 Prepared Data Verification

**Objective:** Verify the numerical integrity, dimensional consistency, and categorical representation of the prepared baseline feature matrices before using them for model development.

The verification will examine whether all prepared values are finite, whether the standardized numerical feature has the expected training-set properties, whether category encoding preserves the intended one-hot representation, and whether the categorical encoder encountered previously unseen categories in the validation or test partitions.

These checks assess the prepared outputs without fitting additional transformations or modifying the underlying data.

In [18]:
# ============================================================
# 3.5 — Prepared Data Verification
# ============================================================

import numpy as np
from scipy import sparse

# Convert only if necessary for numerical inspection
def as_dense(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else matrix

# Inspect the prepared matrices
for name, matrix in [
    ("Training", X_train_prepared),
    ("Validation", X_valid_prepared),
    ("Test", X_test_prepared)
]:
    values = as_dense(matrix)

    print(f"\n{name}:")
    print("Shape:", values.shape)
    print("All values finite:", np.isfinite(values).all())

    # The first output column is standardized transaction amount
    print("Scaled amount mean:", values[:, 0].mean())
    print("Scaled amount std:", values[:, 0].std())

    # Remaining columns represent one-hot-encoded categories
    category_values = values[:, 1:]
    category_row_sums = category_values.sum(axis=1)

    print("Category indicator row sums:",
          np.unique(category_row_sums, return_counts=True))

# Verify categories learned exclusively from training
encoder = (
    preprocessor
    .named_transformers_["categorical"]
    .named_steps["encoder"]
)

training_categories = set(encoder.categories_[0])

print("\nNumber of learned categories:",
      len(training_categories))

for name, partition in [
    ("Validation", valid_df),
    ("Test", test_df)
]:
    unseen = set(partition["category"].unique()) - training_categories
    print(f"Unseen categories in {name}:", sorted(unseen))


Training:
Shape: (396332, 16)
All values finite: True
Scaled amount mean: 7.587115998043922e-17
Scaled amount std: 0.9999999999999999
Category indicator row sums: (array([1.]), array([396332], dtype=int64))

Validation:
Shape: (98194, 16)
All values finite: True
Scaled amount mean: -0.007082159767467827
Scaled amount std: 0.8886192301136145
Category indicator row sums: (array([1.]), array([98194], dtype=int64))

Test:
Shape: (100117, 16)
All values finite: True
Scaled amount mean: -0.012404111069936368
Scaled amount std: 0.8680310041973693
Category indicator row sums: (array([1.]), array([100117], dtype=int64))

Number of learned categories: 15
Unseen categories in Validation: []
Unseen categories in Test: []


**Findings and Interpretation**

The prepared baseline feature matrices passed the numerical integrity and categorical representation checks.

All values are finite across the training, validation, and test partitions. The matrices contain 16 consistently defined features and preserve all 396,332 training, 98,194 validation, and 100,117 test observations.

The standardized transaction-amount feature has a training-set mean effectively equal to zero and a standard deviation of approximately one, confirming the expected behavior of `StandardScaler`.

The validation and test partitions have standardized amount means of approximately −0.0071 and −0.0124, respectively, with standard deviations of approximately 0.8886 and 0.8680. These differences are consistent with applying the training-fitted transformation to later observations without refitting.

The categorical encoder learned all 15 transaction categories from the training partition. Every observation in all three partitions has exactly one active category indicator, and no previously unseen categories were detected in the validation or test partitions.

These results confirm that the baseline preprocessing produces finite, dimensionally consistent, and correctly encoded feature matrices.

**Decision:** The prepared baseline data is suitable for initial model development. No additional cleaning or transformation is required by the current verification results.

This conclusion concerns the integrity of the prepared data, not the eventual predictive performance or operational validity of the fraud-detection model.